# Module 39: Building a Text Classifier — End-to-End NLP Project

Build a sentiment classifier from scratch: tokenizer, transformer encoder, training loop, evaluation, and inference.

**No pretrained models** — every component is implemented from scratch so you understand the full pipeline.

| Input | Output |
|-------|--------|
| `"This movie was fantastic"` | `positive (0.94)` |
| `"Terrible acting, awful"` | `negative (0.91)` |
| `"The movie was okay"` | `neutral (0.72)` |

In [ ]:
import random
import time
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

print(f"PyTorch {torch.__version__}")
torch.manual_seed(42)
random.seed(42)

## 1. Build a Tokenizer from Scratch

Tokenization converts raw text into integer sequences. We build a word-level tokenizer with special tokens.

In [ ]:
import re
from collections import Counter

PAD_TOKEN, UNK_TOKEN, CLS_TOKEN, SEP_TOKEN = "[PAD]", "[UNK]", "[CLS]", "[SEP]"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, CLS_TOKEN, SEP_TOKEN]


class WordTokenizer:
    _WORD_RE = re.compile(r"\w+|[^\w\s]")

    def __init__(self, max_vocab_size: int = 10_000):
        self.max_vocab_size = max_vocab_size
        self.token_to_id: dict[str, int] = {}
        self.id_to_token: dict[int, str] = {}
        for i, tok in enumerate(SPECIAL_TOKENS):
            self.token_to_id[tok] = i
            self.id_to_token[i] = tok

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[PAD_TOKEN]

    @property
    def cls_id(self) -> int:
        return self.token_to_id[CLS_TOKEN]

    def tokenize(self, text: str) -> list[str]:
        return self._WORD_RE.findall(text.lower())

    def build_vocab(self, texts: list[str]) -> None:
        counter: Counter = Counter()
        for t in texts:
            counter.update(self.tokenize(t))
        budget = self.max_vocab_size - len(SPECIAL_TOKENS)
        for word, _ in counter.most_common(budget):
            if word not in self.token_to_id:
                idx = len(self.token_to_id)
                self.token_to_id[word] = idx
                self.id_to_token[idx] = word

    def encode(self, text: str, max_length: int = 128) -> list[int]:
        tokens = self.tokenize(text)
        ids = [self.token_to_id.get(t, self.token_to_id[UNK_TOKEN]) for t in tokens]
        ids = [self.cls_id] + ids
        if len(ids) > max_length:
            ids = ids[:max_length]
        else:
            ids += [self.pad_id] * (max_length - len(ids))
        return ids

    def decode(self, ids: list[int]) -> str:
        tokens = [
            self.id_to_token.get(i, UNK_TOKEN)
            for i in ids
            if self.id_to_token.get(i, UNK_TOKEN) not in SPECIAL_TOKENS
        ]
        return " ".join(tokens)


print("WordTokenizer defined.")

### Inspect the Vocabulary

The vocabulary maps tokens to integer IDs. Special tokens come first.

In [ ]:
# Encode / decode round-trip
test_text = "This was a fantastic movie!"
encoded = tokenizer.encode(test_text, max_length=12)
decoded = tokenizer.decode(encoded)
print(f"Original: '{test_text}'")
print(f"Encoded:  {encoded}")
print(f"Decoded:  '{decoded}'")
print(f"\nPad ID: {tokenizer.pad_id}  CLS ID: {tokenizer.cls_id}")
print(f"Non-pad tokens: {sum(1 for x in encoded if x != tokenizer.pad_id)}")

## 2. Tokenize Sample Texts

In [ ]:
sample_texts = [
    "This movie was absolutely fantastic!",
    "Terrible acting and awful script.",
    "The film was okay, nothing special.",
]

tokenizer = WordTokenizer(max_vocab_size=200)
tokenizer.build_vocab(sample_texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Vocabulary: {tokenizer.token_to_id}\n")

for text in sample_texts:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text, max_length=15)
    decoded = tokenizer.decode(ids)
    print(f"Text:    {text}")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print(f"Decoded: {decoded}")
    print()

### Shape Walkthrough

Trace data shapes through the model layer by layer.

In [ ]:
# Shape walkthrough with a single sample
x = dummy_ids[:1]  # (1, 16)
m = dummy_mask[:1]

emb = demo_model.embedding(x) * math.sqrt(64)
print(f"1. input_ids:              {list(x.shape)}")
print(f"2. After embedding:        {list(emb.shape)}")

emb_pos = demo_model.pos_encoder(emb)
print(f"3. After positional enc:   {list(emb_pos.shape)}")

enc_out = demo_model.encoder(emb_pos, src_key_padding_mask=(m == 0))
print(f"4. After transformer enc:  {list(enc_out.shape)}")

pooled = enc_out[:, 0]  # CLS pooling
print(f"5. After CLS pooling:      {list(pooled.shape)}")

logit = demo_model.classifier(pooled)
print(f"6. After classifier head:  {list(logit.shape)}")

## 3. Build the TransformerTextClassifier

Architecture: Embedding + Positional Encoding + TransformerEncoder + CLS Pooling + Classification Head

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerTextClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, num_classes=3, max_len=512,
                 dropout=0.1, padding_idx=0, pool="cls"):
        super().__init__()
        self.pool = pool
        self.d_model = d_model
        self.padding_idx = padding_idx

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, num_classes))

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids, attention_mask=None):
        if attention_mask is None:
            attention_mask = (input_ids != self.padding_idx).float()
        src_key_padding_mask = attention_mask == 0

        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)

        if self.pool == "cls":
            pooled = x[:, 0]
        else:
            mask_exp = attention_mask.unsqueeze(-1)
            pooled = (x * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)

        return self.classifier(pooled)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("TransformerTextClassifier defined.")

In [ ]:
# Quick forward pass test
demo_model = TransformerTextClassifier(vocab_size=200, d_model=64, nhead=4,
                                        num_layers=2, dim_feedforward=128,
                                        num_classes=3, max_len=32, pool="cls")
print(f"Parameters: {demo_model.count_parameters():,}\n")

dummy_ids = torch.randint(1, 200, (4, 16))
dummy_ids[:, 0] = 2  # CLS
dummy_mask = (dummy_ids != 0).float()

with torch.no_grad():
    logits = demo_model(dummy_ids, dummy_mask)

print(f"Input shape:  {list(dummy_ids.shape)}")
print(f"Output shape: {list(logits.shape)}")
print(f"Logits[0]:    {logits[0].tolist()}")
print(f"Probs[0]:     {torch.softmax(logits[0], dim=-1).tolist()}")

## 4. Create Synthetic Dataset

Generate a simple sentiment dataset using templates with sentiment-bearing adjectives.

### Inspect the Data

Look at a few samples and check the label distribution.

In [ ]:
# Show a few training samples
print("Sample training data:")
for i in range(min(5, len(train_set))):
    item = train_set[i]
    ids_list = item["input_ids"].tolist()
    decoded = tokenizer.decode(ids_list)
    label = LABEL_NAMES[item["label"].item()]
    non_pad = sum(1 for x in ids_list if x != 0)
    print(f"  [{label:>8s}] ({non_pad:2d} tokens) {decoded}")

In [ ]:
_POS_TMPLS = [
    "This was absolutely {adj}", "I loved {adj} every moment",
    "A {adj} experience overall", "Really {adj} and well done",
    "{adj} movie I highly recommend", "The acting was {adj} and the story was {adj}",
]
_POS_ADJS = ["great", "fantastic", "wonderful", "amazing", "excellent",
             "brilliant", "outstanding", "superb", "incredible", "perfect"]

_NEG_TMPLS = [
    "This was absolutely {adj}", "I found it {adj} and boring",
    "A {adj} waste of time", "Really {adj} do not recommend",
    "{adj} movie with {adj} acting", "The acting was {adj} and the plot was {adj}",
]
_NEG_ADJS = ["terrible", "horrible", "awful", "dreadful", "atrocious",
             "pathetic", "abysmal", "lousy", "appalling", "miserable"]

_NEU_TMPLS = [
    "The movie was {adj}", "It was {adj} nothing more",
    "A {adj} film overall", "Not bad not great just {adj}",
    "{adj} movie with {adj} moments", "Pretty {adj} if you ask me",
]
_NEU_ADJS = ["okay", "average", "mediocre", "decent", "passable",
             "ordinary", "unremarkable", "fair", "moderate", "standard"]

LABEL_NAMES = ["positive", "negative", "neutral"]


def generate_data(n_per_class=200, seed=42):
    rng = random.Random(seed)
    texts, labels = [], []
    for tmpls, adjs, label in [
        (_POS_TMPLS, _POS_ADJS, 0),
        (_NEG_TMPLS, _NEG_ADJS, 1),
        (_NEU_TMPLS, _NEU_ADJS, 2),
    ]:
        for _ in range(n_per_class):
            tmpl = rng.choice(tmpls)
            text = tmpl.replace("{adj}", rng.choice(adjs), 1)
            text = text.replace("{adj}", rng.choice(adjs))
            texts.append(text)
            labels.append(label)
    combined = list(zip(texts, labels))
    rng.shuffle(combined)
    texts, labels = zip(*combined)
    return list(texts), list(labels)


texts, labels = generate_data(n_per_class=200)
print(f"Total samples: {len(texts)}")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {labels.count(i)}")
print(f"\nExample: '{texts[0]}' -> {LABEL_NAMES[labels[0]]}")

## 5. Dataset and DataLoader Setup

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = [tokenizer.encode(t, max_length) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings[idx], dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate_fn(batch):
    input_ids = nn.utils.rnn.pad_sequence(
        [item["input_ids"] for item in batch], batch_first=True, padding_value=0,
    )
    labels = torch.stack([item["label"] for item in batch])
    attention_mask = (input_ids != 0).float()
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# Build tokenizer on full data, then split
tokenizer = WordTokenizer(max_vocab_size=500)
tokenizer.build_vocab(texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")

max_length = 32
dataset = TextDataset(texts, labels, tokenizer, max_length)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_set, val_set, test_set = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)
print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=64, collate_fn=collate_fn)

# Inspect one batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {list(v.shape)}")

## 6. Training Loop

Full training with AdamW, OneCycleLR, gradient clipping, BF16 autocast, and early stopping.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, max_grad_norm=1.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        ids, mask, lbls = batch["input_ids"], batch["attention_mask"], batch["labels"]
        with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
            logits = model(ids, mask)
            loss = criterion(logits, lbls)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * lbls.size(0)
        correct += (logits.argmax(-1) == lbls).sum().item()
        total += lbls.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        ids, mask, lbls = batch["input_ids"], batch["attention_mask"], batch["labels"]
        logits = model(ids, mask)
        loss = criterion(logits, lbls)
        total_loss += loss.item() * lbls.size(0)
        preds = logits.argmax(-1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
        all_preds.extend(preds.tolist())
        all_labels.extend(lbls.tolist())
    return total_loss / total, correct / total, all_preds, all_labels


print("Training functions defined.")

In [ ]:
# Create model
model = TransformerTextClassifier(
    vocab_size=tokenizer.vocab_size, d_model=64, nhead=4, num_layers=2,
    dim_feedforward=128, num_classes=3, max_len=max_length,
    dropout=0.1, padding_idx=tokenizer.pad_id, pool="cls",
)
print(f"Parameters: {model.count_parameters():,}")

num_epochs = 15
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, epochs=num_epochs, steps_per_epoch=len(train_loader),
)

# Training loop with early stopping
best_val_loss = float("inf")
patience, patience_ctr = 4, 0
best_state = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, num_epochs + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler)
    v_loss, v_acc, _, _ = evaluate(model, val_loader, criterion)

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:2d}/{num_epochs} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss:.4f} Acc: {v_acc:.4f} | LR: {lr:.6f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        patience_ctr = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

if best_state:
    model.load_state_dict(best_state)
    print(f"Loaded best model (val loss: {best_val_loss:.4f})")

## 7. Training Curves (Print-Based)

In [ ]:
def print_curve(values, label, width=50):
    mn, mx = min(values), max(values)
    rng = mx - mn if mx > mn else 1.0
    print(f"\n{label}")
    print(f"{'Epoch':>6s} {'Value':>8s}  {'Bar'}")
    for i, v in enumerate(values, 1):
        bar_len = int((v - mn) / rng * width)
        print(f"{i:>6d} {v:>8.4f}  {'#' * bar_len}")

print_curve(history["train_loss"], "Train Loss")
print_curve(history["val_loss"], "Val Loss")
print_curve(history["train_acc"], "Train Accuracy")
print_curve(history["val_acc"], "Val Accuracy")

## 8. Evaluation: Accuracy, F1, Confusion Matrix

In [ ]:
def compute_metrics(preds, labels, num_classes, label_names):
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for p, l in zip(preds, labels):
        confusion[l][p] += 1
    per_class = {}
    f1s = []
    for c in range(num_classes):
        tp = confusion[c][c].item()
        fp = confusion[:, c].sum().item() - tp
        fn = confusion[c, :].sum().item() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        per_class[label_names[c]] = {"precision": prec, "recall": rec, "f1": f1,
                                      "support": confusion[c].sum().item()}
        f1s.append(f1)
    return {"confusion": confusion, "per_class": per_class, "macro_f1": sum(f1s)/len(f1s)}


# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f}")

metrics = compute_metrics(test_preds, test_labels, 3, LABEL_NAMES)

# Confusion matrix
print("\nConfusion Matrix:")
header = "          " + "".join(f"{n:>12s}" for n in LABEL_NAMES)
print(header)
print("          " + "-" * (12 * 3))
for i in range(3):
    row = f"{LABEL_NAMES[i]:>9s} |"
    for j in range(3):
        row += f"{metrics['confusion'][i][j].item():>12d}"
    print(row)

# Classification report
print(f"\n{'':>12s} {'Prec':>8s} {'Recall':>8s} {'F1':>8s} {'Support':>8s}")
print("-" * 48)
for name, m in metrics["per_class"].items():
    print(f"{name:>12s} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['support']:>8d}")
print("-" * 48)
print(f"{'Macro F1':>12s} {'':>8s} {'':>8s} {metrics['macro_f1']:>8.4f}")

### Batch Inference

Process multiple texts at once for better throughput.

In [ ]:
# Batch inference
def predict_batch(texts, model, tokenizer, label_names, max_length=32):
    model.eval()
    encoded = [tokenizer.encode(t, max_length) for t in texts]
    ids = nn.utils.rnn.pad_sequence(
        [torch.tensor(e) for e in encoded], batch_first=True, padding_value=0,
    )
    mask = (ids != 0).float()
    with torch.no_grad():
        logits = model(ids, mask)
    probs = torch.softmax(logits, dim=-1)
    confs, preds = probs.max(dim=-1)
    return [
        {"text": t, "label": label_names[p.item()], "confidence": c.item()}
        for t, p, c in zip(texts, preds, confs)
    ]


batch_results = predict_batch(test_texts, model, tokenizer, LABEL_NAMES)
print("Batch Inference Results:")
for r in batch_results:
    print(f"  {r['label']:>8s} ({r['confidence']:.3f})  '{r['text']}'")

## 9. Inference Pipeline: Text -> Prediction

In [ ]:
def predict(text, model, tokenizer, label_names, max_length=32):
    model.eval()
    ids = torch.tensor([tokenizer.encode(text, max_length)])
    mask = (ids != 0).float()
    with torch.no_grad():
        logits = model(ids, mask)
    probs = torch.softmax(logits, dim=-1)
    conf, pred = probs.max(dim=-1)
    return {
        "text": text,
        "label": label_names[pred.item()],
        "confidence": conf.item(),
        "probs": {n: probs[0][i].item() for i, n in enumerate(label_names)},
    }


test_texts = [
    "This was absolutely wonderful and amazing",
    "Terrible movie with horrible acting",
    "The film was okay nothing special",
    "I loved every moment of this masterpiece",
    "Dreadful experience would not recommend",
    "A decent movie with some good parts",
]

print("Inference Results:")
print("=" * 70)
for text in test_texts:
    r = predict(text, model, tokenizer, LABEL_NAMES)
    probs_str = ", ".join(f"{k}: {v:.3f}" for k, v in r["probs"].items())
    print(f"  '{r['text']}'")
    print(f"    -> {r['label']} (confidence: {r['confidence']:.3f})")
    print(f"    -> [{probs_str}]")
    print()

### Compiled vs Eager: Verify Outputs Match

Sanity check that torch.compile doesn't change model outputs.

In [ ]:
try:
    model.eval()
    with torch.no_grad():
        eager_out = model(dummy_ids, dummy_mask)
        compiled_out = compiled(dummy_ids, dummy_mask)
    match = torch.allclose(eager_out, compiled_out, atol=1e-5)
    max_diff = (eager_out - compiled_out).abs().max().item()
    print(f"Outputs match: {match}")
    print(f"Max absolute difference: {max_diff:.2e}")
except NameError:
    print("Compiled model not available (torch.compile was skipped)")

## 10. torch.compile for Serving

In [ ]:
dummy_ids = torch.randint(1, tokenizer.vocab_size, (1, max_length))
dummy_ids[:, 0] = tokenizer.cls_id
dummy_mask = torch.ones(1, max_length)

def benchmark(model, ids, mask, n_warmup=5, n_runs=50):
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            model(ids, mask)
        start = time.perf_counter()
        for _ in range(n_runs):
            model(ids, mask)
        return (time.perf_counter() - start) / n_runs * 1000

eager_ms = benchmark(model, dummy_ids, dummy_mask)
print(f"Eager inference: {eager_ms:.2f} ms")

try:
    compiled = torch.compile(model, mode="reduce-overhead")
    comp_ms = benchmark(compiled, dummy_ids, dummy_mask, n_warmup=10)
    print(f"Compiled inference: {comp_ms:.2f} ms")
    print(f"Speedup: {eager_ms / comp_ms:.2f}x")
except Exception as e:
    print(f"torch.compile skipped: {e}")

## 11. Exercise: Add a Third Class (Neutral) and Retrain

**Challenge**: Our model already handles 3 classes. Try these modifications:

1. **Increase neutral difficulty**: Add templates where neutral and positive/negative overlap (e.g., "The movie had good moments but was ultimately average")
2. **Add a 4th class**: Add "mixed" sentiment ("The acting was great but the plot was terrible")
3. **Try mean pooling**: Change `pool="cls"` to `pool="mean"` and compare performance
4. **Experiment with model size**: Try `d_model=32` vs `d_model=128` — how does capacity affect accuracy?

```python
# Template for adding a 4th class:
_MIXED_TMPLS = [
    "The acting was {pos_adj} but the plot was {neg_adj}",
    "{pos_adj} visuals but {neg_adj} story",
]
LABEL_NAMES = ["positive", "negative", "neutral", "mixed"]
# Update num_classes=4 in the model
```

## Key Takeaways

1. **Tokenization is the foundation** — word-level tokenization with special tokens (`[PAD]`, `[UNK]`, `[CLS]`) converts text into model-ready integer sequences
2. **`nn.Embedding` is a learnable lookup table** — `padding_idx` ensures padding tokens don't contribute to the model
3. **Positional encoding adds sequence order** — sinusoidal encodings give the transformer position awareness
4. **Pre-norm transformers train more stably** — `norm_first=True` is the modern default
5. **Dynamic padding in `collate_fn`** — pad to the longest sequence per-batch, not globally
6. **Full training recipe**: AdamW + OneCycleLR + gradient clipping + early stopping + BF16
7. **Metrics beyond accuracy**: per-class precision/recall/F1 and confusion matrix reveal model weaknesses
8. **torch.compile for serving** — compile the trained model for faster inference with zero accuracy change
9. **End-to-end ownership** — building every component from scratch gives deep understanding of the NLP pipeline